In [5]:
import spaTrack as spt
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, rgb2hex
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
import numpy as np
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
import numpy as np
from sklearn.metrics import jaccard_score
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42 # ADOBE AI 字帖
import os
from matplotlib.font_manager import fontManager, FontProperties
import gc
from scipy.sparse import csr_matrix
fontManager.addfont('/data/work/Arial.ttf')

font = FontProperties(fname='/data/work/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name
sc.settings.verbosity = 0

In [8]:
adata

AnnData object with n_obs × n_vars = 2014 × 12212
    obs: 'region', 'slices', 'z', 'cluster', 'level1', 'level0', 'level-1', 'new_cluster', 'anno_whole_cluster', 'ax', 'ay', 'az', 'entropy', 'tran', 'ptime'
    uns: 'E_grid', 'P_grid', 'V_grid', 'entropy value', 'entropy value order', 'pca', 'anno_whole_cluster_colors'
    obsm: '3d_align_spatial_confine', 'X_pca', 'X_spatial', 'spatial', 'velocity_spatial'
    varm: 'PCs'
    obsp: 'trans', 'trans_neigh_csr'

In [9]:
colormap = {
    'Ob-Neu-1': '#8963AD',
'Ob-Neu-2': '#8CC494',
'Ob-Neu-3': '#DAB063',
'Ob-Neu-4': '#FE972B',
'Ob-Neu-5': '#EB4647',
'Ob-Neu-6': '#704599',
'Ob-Neu-7': '#84BF96',
'Ob-Neu-8': '#F16868',
'Ob-Glia-1': '#936FB3',
'Ob-Neu-15': '#B49C72',
'Ob-Neu-9': '#E2C26F',
'Ob-Neu-10': '#9CD47A',
'Ob-Neu-11': '#845DAA',
'Ob-Glia-2': '#F8A05F',
'Ob-Neu-12': '#D5A6A5',
'Ob-Neu-13': '#B79ACA',
'Ob-Neu-14': '#1F78B3',
'Ob-Glia-3': '#8FCD70',
'Ob-Vasc-1': '#AE87FF',
'Epend-1': '#90C0DB',
'Epend-2': '#C99B7E',
'Epend-3': '#FBB268',
'Epend-4': '#6A9E4A',
'Epend-5': '#AA9C6C',
'Epend-6': '#ABDA8B',
'Epend-7': '#2C80B8',
'Epend-8': '#FE8002',
}

In [2]:
adatas = []
for i in ['11', '12', '13', '27', '28', '29']:
    adata_ = sc.read_h5ad(f'/data/work/22.fusemap/08.spatrack/script_20251011/ob_with_epend/{i}.h5ad')
    # sc.pl.embedding(adata, basis = '3d_align_spatial_confine', color = 'anno_whole_cluster', title = str(i))
    adatas.append(adata_)
adata = ad.concat(adatas)
adata.obs['celltype'] = adata.obs['anno_whole_cluster'].tolist()

In [2]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy
import seaborn as sns
import warnings
from IPython.display import display

def assess_start_cluster(adata, obs_key):
    cluster_name_list=list()
    entropy_list=list()
    cell_name_list=list()
    from scipy.sparse import issparse

    if issparse(adata.X):
        adata.X = adata.X.toarray()
    for i in range(0,len(adata.obs.index)):
        cell_id=adata.obs.index[i]
        cluster_name=adata.obs[obs_key][i]
        adata_cluster=adata[adata.obs.index.isin([cell_id])]
        matrix = np.array(adata_cluster.X)
        entropy_name=entropy(matrix[0])
        cluster_name_list.append(cluster_name)
        cell_name_list.append(cell_id)
        entropy_list.append(entropy_name)
    df_value=pd.DataFrame({'cluster_name': cluster_name_list,'cell_name':cell_name_list,'entropy':entropy_list})
    adata.obs['entropy']=df_value['entropy'].values
    df_obs=adata.obs
    cluster_order=list(pd.DataFrame(df_obs[[obs_key,'entropy']].groupby([obs_key]).mean()).sort_values(['entropy'],ascending=False).index)
    df_entropy=pd.DataFrame(df_obs[[obs_key,'entropy']].groupby([obs_key]).mean()).sort_values(['entropy'],ascending=False)
    df_obs[obs_key]= pd.Categorical(df_obs[obs_key], categories=cluster_order, ordered=True)
    print('Cluster order sorted by entropy value: ',list(df_entropy.index))
    df_entropy.index=list(df_entropy.index)
    adata.uns['entropy value']=df_obs
    adata.uns['entropy value order']=df_entropy
        #display(df_entropy)
    return adata

In [4]:
adata.X = adata.X.astype(int)
adata=assess_start_cluster(adata, 'celltype')

Cluster order sorted by entropy value:  ['Epend-6', 'Ob-Neu-8', 'Ob-Neu-14', 'Ob-Neu-4', 'Ob-Glia-3', 'Ob-Neu-10', 'Epend-1', 'Epend-8', 'Ob-Neu-11', 'Ob-Neu-2', 'Ob-Neu-5', 'Ob-Neu-7', 'Ob-Neu-3', 'Ob-Glia-1', 'Epend-7', 'Ob-Neu-1', 'Ob-Neu-13', 'Epend-5', 'Ob-Neu-12', 'Ob-Neu-6', 'Epend-4', 'Ob-Neu-9', 'Epend-2', 'Ob-Vasc-1', 'Ob-Glia-2']


In [12]:
for slice_code in ['11', '12', '13', '27', '28', '29']:
    adata = sc.read_h5ad(f'/data/work/22.fusemap/08.spatrack/script_20251011/ob_with_epend/{slice_code}.h5ad')
    adata.X = adata.X.astype(int)
    adata=assess_start_cluster(adata, 'anno_whole_cluster')
    adata.obs['cluster'] = adata.obs['anno_whole_cluster'].copy()

    adata.obsm['X_spatial'] = adata.obsm['3d_align_spatial_confine'][:,0:2]
    start_cells = spt.set_start_cells(adata, select_way='coordinates', start_point=adata[adata.obs['cluster'].isin(['Epend-5', 'Epend-7','Epend-6'])].obsm['X_spatial'] )
    adata.obsp["trans"] = spt.get_ot_matrix(adata, data_type="spatial",alpha1=0.5,alpha2=0.5)
    adata.obs["ptime"] = spt.get_ptime(adata, start_cells)
    adata.uns["E_grid"], adata.uns["V_grid"] = spt.get_velocity(adata, basis="spatial", n_neigh_pos=100,n_neigh_gene=0)


    adata.write(f'/data/work/22.fusemap/08.spatrack/script_20251011/tracks_plot_03/data/{slice_code}.h5ad')

Cluster order sorted by entropy value:  ['Ob-Neu-8', 'Ob-Glia-3', 'Ob-Neu-10', 'Ob-Neu-14', 'Ob-Neu-4', 'Ob-Neu-11', 'Ob-Neu-7', 'Ob-Neu-1', 'Ob-Neu-2', 'Ob-Neu-5', 'Ob-Neu-3', 'Ob-Neu-6', 'Epend-5', 'Ob-Neu-9', 'Ob-Neu-13', 'Ob-Neu-12', 'Epend-2', 'Epend-4', 'Ob-Glia-2', 'Epend-7', 'Ob-Glia-1', 'Ob-Vasc-1']
X_pca is not in adata.obsm, automatically do PCA first.
The velocity of cells store in 'velocity_spatial'.
Cluster order sorted by entropy value:  ['Ob-Neu-8', 'Ob-Neu-10', 'Ob-Neu-14', 'Ob-Neu-5', 'Ob-Neu-7', 'Epend-2', 'Ob-Neu-4', 'Ob-Neu-1', 'Ob-Neu-11', 'Epend-7', 'Ob-Neu-3', 'Ob-Neu-2', 'Ob-Neu-12', 'Ob-Neu-13', 'Ob-Glia-1', 'Ob-Neu-6', 'Ob-Glia-3', 'Ob-Vasc-1', 'Ob-Glia-2', 'Ob-Neu-9', 'Epend-5', 'Epend-4']
X_pca is not in adata.obsm, automatically do PCA first.
The velocity of cells store in 'velocity_spatial'.
Cluster order sorted by entropy value:  ['Ob-Neu-14', 'Ob-Neu-8', 'Epend-8', 'Ob-Neu-10', 'Epend-2', 'Ob-Neu-4', 'Ob-Neu-5', 'Ob-Neu-7', 'Epend-7', 'Ob-Glia-3', 'Ob-N

In [14]:
1

1